# Analytics Reliability & Release Control

## tl;dr

The candidate release decision is **SHIP WITH CAVEAT**. Critical controls pass at **98.9%**, freshness SLA compliance is **99.2%**, and one accepted high-severity exception remains open.


## Context & Methods

Decision: ship, hold, or ship with a caveat. The raw layer contains five disclosed, deterministic anomaly scenarios. Release status is a gate, not a weighted trust score.

### Key Assumptions

- Critical failures always block.
- A high warning can ship only with an owner, future expiry and accepted exception.
- Every release records the quality-evidence date used to derive its failure and warning counts.
- The latest complete synthetic snapshot is 30 Jun 2026.


In [1]:
from pathlib import Path
import csv, json
PROJECT_DIR = Path.cwd()
def read_csv(name):
    with (PROJECT_DIR / name).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))
summary = json.loads((PROJECT_DIR / 'data/curated/summary.json').read_text())
failures = read_csv('data/curated/failure_evidence.csv')
print(json.dumps(summary, indent=2))
print(f"reviewable failure/warning rows={len(failures)}")


{
  "release_decision": "SHIP WITH CAVEAT",
  "critical_pass_rate": 0.988889,
  "freshness_sla_rate": 0.991667,
  "latest_reconciliation_variance": 0.00215,
  "open_high_critical_incidents": 1,
  "average_change_lead_days": 4.33
}
reviewable failure/warning rows=20


## Data

The control plane uses one row per scheduled asset run and one row per run × quality rule. Raw records remain inspectable and intentional scenario anomalies are separately declared.


In [2]:
quality = read_csv('data/raw/quality_results.csv')
status_counts = {}
for row in quality:
    status_counts[row['status']] = status_counts.get(row['status'], 0) + 1
print(f"quality results={len(quality):,} | status counts={status_counts}")


quality results=2,928 | status counts={'PASS': 2908, 'FAIL': 11, 'WARN': 9}


## Results

Failures are localized by asset, date, rule, severity and owner. This makes a release decision inspectable instead of relying on a cosmetic health score.


In [3]:
for row in failures[:12]:
    print(f"{row['run_date']} | {row['asset_id']:<18} | {row['status']:<4} | {row['rule_name']} | observed={row['observed_value']} threshold={row['threshold']}")


2026-06-30 | CAMPAIGN_SPEND     | WARN | Unknown taxonomy rate | observed=0.02 threshold=0.01
2026-06-29 | CAMPAIGN_SPEND     | WARN | Unknown taxonomy rate | observed=0.02 threshold=0.01
2026-06-28 | CAMPAIGN_SPEND     | WARN | Unknown taxonomy rate | observed=0.02 threshold=0.01
2026-06-18 | COMMERCE_ORDERS    | FAIL | Raw-to-dashboard variance | observed=0.021 threshold=0.005
2026-06-18 | COMMERCE_ORDERS    | WARN | Required type validity | observed=4.0 threshold=0.0
2026-06-17 | COMMERCE_ORDERS    | FAIL | Raw-to-dashboard variance | observed=0.021 threshold=0.005
2026-06-17 | COMMERCE_ORDERS    | WARN | Required type validity | observed=4.0 threshold=0.0
2026-06-03 | CRM_LEADS          | FAIL | Partition row coverage | observed=0.58 threshold=0.9
2026-06-03 | CRM_LEADS          | FAIL | Raw-to-dashboard variance | observed=0.421 threshold=0.005
2026-06-03 | CRM_LEADS          | WARN | Freshness lag | observed=520.0 threshold=240.0
2026-05-10 | CAMPAIGN_SPEND     | FAIL | Raw-to-da

## Takeaways

1. The synthetic release can ship only with the documented, owned high-severity caveat.
2. Critical uniqueness, completeness and reconciliation failures remain hard blockers.
3. Each anomaly is traceable from raw evidence through rule result, incident and remediation release.
